# Standard inpainting + target-object attention map

This notebook uses a normal Diffusers inpainting pipeline to add a person on the sofa, then estimates a simple image-to-text attention map for the target noun using CLIP patch embeddings and Matplotlib overlays.


In [ ]:
import os
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw

from diffusers import AutoPipelineForInpainting
from transformers import CLIPProcessor, CLIPTextModel, CLIPVisionModel

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "diffusers/stable-diffusion-xl-1.0-inpainting-0.1"
CLIP_ID = "openai/clip-vit-base-patch32"

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")


In [ ]:
def make_scene():
    width, height = 768, 512
    canvas = Image.new("RGB", (width, height), (235, 230, 220))
    draw = ImageDraw.Draw(canvas)

    # Scene with a sofa and a clear inpaint area
    draw.rounded_rectangle((120, 260, 650, 470), radius=28, fill=(170, 150, 130))
    draw.rounded_rectangle((200, 220, 600, 300), radius=20, fill=(185, 165, 145))
    draw.rectangle((150, 270, 620, 350), fill=(150, 130, 110))

    target_box = (300, 180, 470, 380)
    draw.rounded_rectangle(target_box, radius=18, outline=(80, 80, 80), width=3, fill=(255, 255, 255))
    return canvas, target_box


source, target_box = make_scene()
mask = Image.new("L", source.size, 0)
mask_draw = ImageDraw.Draw(mask)
mask_draw.rounded_rectangle(target_box, radius=18, fill=255)

source.save("source_scene.png")
mask.save("inpaint_mask.png")

pipe = AutoPipelineForInpainting.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    safety_checker=None,
)
pipe = pipe.to(DEVICE)

prompt = (
    "Add a realistic person sitting on the sofa in the empty area, natural pose, "
    "photorealistic lighting, high detail, natural clothes, cohesive scene composition."
)

result = pipe(
    prompt=prompt,
    image=source,
    mask_image=mask,
    num_inference_steps=25,
    guidance_scale=7.5,
    strength=0.85,
).images[0].convert("RGB")

result.save("inpainted_person_on_sofa.png")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(source)
axes[0].set_title("Source scene")
axes[0].axis("off")
axes[1].imshow(result)
axes[1].set_title("General inpaint: add a person on the sofa")
axes[1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
def attention_map_for_target(image: Image.Image, text_prompt: str, target_word: str, device: str = DEVICE):
    processor = CLIPProcessor.from_pretrained(CLIP_ID)
    vision_model = CLIPVisionModel.from_pretrained(CLIP_ID).to(device)
    text_model = CLIPTextModel.from_pretrained(CLIP_ID).to(device)
    vision_model.eval(); text_model.eval()

    inputs = processor(images=image, text=text_prompt, return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        text_outputs = text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            output_hidden_states=True,
            return_dict=True,
        )
        vision_outputs = vision_model(
            pixel_values=inputs["pixel_values"],
            output_hidden_states=True,
            return_dict=True,
        )

    patch_tokens = vision_outputs.last_hidden_state[:, 1:, :]
    patch_tokens = patch_tokens.reshape(1, 7, 7, patch_tokens.shape[-1])
    patch_flat = patch_tokens[0].reshape(-1, patch_tokens.shape[-1])
    patch_flat = patch_flat / patch_flat.norm(dim=-1, keepdim=True).clamp_min(1e-8)

    tokenizer = processor.tokenizer
    token_ids = inputs["input_ids"][0]
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    target_idx = None
    for idx, token in enumerate(tokens):
        if target_word.lower() in token.lower():
            target_idx = idx
            break
    if target_idx is None:
        raise ValueError(f"Could not find target token '{target_word}' in the prompt tokens: {tokens}")

    text_embedding = text_outputs.last_hidden_state[0, target_idx].float()
    text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True).clamp_min(1e-8)

    scores = patch_flat @ text_embedding
    scores = scores.reshape(7, 7)
    heatmap = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    return heatmap.cpu().numpy()


target_word = "person"
text_prompt = "a person sitting on a sofa in a living room"
heatmap = attention_map_for_target(result, text_prompt, target_word, device=DEVICE)
heatmap_resized = np.array(
    Image.fromarray((heatmap * 255).astype(np.uint8)).resize(result.size[::-1], resample=Image.BILINEAR)
)
heatmap_rgb = np.stack([heatmap_resized] * 3, axis=-1)

heat_rgba = np.zeros((*heatmap_resized.shape[:2], 4), dtype=np.uint8)
heat_rgba[:, :, :3] = heatmap_rgb
heat_rgba[:, :, 3] = 128
overlay = Image.blend(result.convert("RGBA"), Image.fromarray(heat_rgba, mode="RGBA"), alpha=0.45)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(result)
axes[0].set_title("General inpaint result")
axes[0].axis("off")
axes[1].imshow(heatmap, cmap="jet")
axes[1].set_title(f"Image-to-text attention: {target_word}")
axes[1].axis("off")
axes[2].imshow(overlay)
axes[2].set_title("Overlay")
axes[2].axis("off")
plt.tight_layout()
plt.show()
